# argentina.geo.direcciones — Georreferenciar direcciones reales

Recorrido **usando el paquete** para georreferenciar un lote de direcciones de puntos icónicos de Argentina contra la API pública [Georef](https://datosgobar.github.io/georef-ar-api/) y plotearlos sobre un mapa Folium con fondo Argenmap del IGN.

Requiere red (el notebook llama a `apis.datos.gob.ar/georef` en cada celda).

## 1. Setup

In [1]:
import argentina as arg
from argentina.geo import direcciones, basemaps
import pandas as pd
import folium

print(f"argentina v{arg.__version__}")
print(f"endpoint Georef: {direcciones.BASE_URL}")

argentina v0.0.20
endpoint Georef: https://apis.datos.gob.ar/georef/api


## 2. Una dirección puntual

Pedimos la Casa Rosada (Balcarce 50, CABA) y vemos qué nos devuelve Georef.

In [2]:
casa_rosada = direcciones.georreferenciar(
    direccion="Balcarce 50",
    provincia="CABA",
)
casa_rosada

{'altura': {'unidad': None, 'valor': 50},
 'calle': {'categoria': 'CALLE', 'id': '0200701001910', 'nombre': 'BALCARCE'},
 'calle_cruce_1': {'categoria': None, 'id': None, 'nombre': None},
 'calle_cruce_2': {'categoria': None, 'id': None, 'nombre': None},
 'departamento': {'id': '02007', 'nombre': 'Comuna 1'},
 'localidad_censal': {'id': '02000010',
  'nombre': 'Ciudad Autónoma de Buenos Aires'},
 'nomenclatura': 'BALCARCE 50, Comuna 1, Ciudad Autónoma de Buenos Aires',
 'piso': None,
 'provincia': {'id': '02', 'nombre': 'Ciudad Autónoma de Buenos Aires'},
 'ubicacion': {'lat': -34.60821175664485, 'lon': -58.37075020911925}}

In [3]:
# Helper directo cuando solo querés coordenadas
direcciones.coordenadas(direccion="Balcarce 50", provincia="CABA")

(-34.60821175664485, -58.37075020911925)

## 3. Lote de puntos icónicos

Georreferenciamos varios lugares conocidos. Filtramos por provincia/localidad para reducir falsos positivos (hay muchas "Av. Belgrano" en el país).

In [4]:
puntos = [
    {'nombre': 'Casa Rosada',           'direccion': 'Balcarce 50',           'provincia': 'CABA',         'localidad': None},
    {'nombre': 'Congreso de la Nación', 'direccion': 'Hipólito Yrigoyen 1849','provincia': 'CABA',         'localidad': None},
    {'nombre': 'Obelisco',              'direccion': 'Av. Corrientes 1066',   'provincia': 'CABA',         'localidad': None},
    {'nombre': 'Catedral de Córdoba',   'direccion': 'Independencia 30',      'provincia': 'Córdoba',      'localidad': 'Córdoba'},
    {'nombre': 'Monumento Bandera',     'direccion': 'Av. Belgrano 1700',     'provincia': 'Santa Fe',     'localidad': 'Rosario'},
    {'nombre': 'Catedral de La Plata',  'direccion': 'Calle 14 esq. 51',      'provincia': 'Buenos Aires', 'localidad': 'La Plata'},
    {'nombre': 'Cerro de la Gloria',    'direccion': 'Av. del Libertador',    'provincia': 'Mendoza',      'localidad': 'Mendoza'},
]

filas = []
for p in puntos:
    coords = direcciones.coordenadas(
        direccion=p['direccion'],
        provincia=p['provincia'],
        localidad=p['localidad'],
    )
    filas.append({
        **p,
        'lat': coords[0] if coords else None,
        'lon': coords[1] if coords else None,
    })

df = pd.DataFrame(filas)
df

,nombre,direccion,provincia,localidad,lat,lon
0,Casa Rosada,Balcarce 50,CABA,None,-34.608212,-58.370750
1,Congreso de la Nación,Hipólito Yrigoyen 1849,CABA,None,-34.610386,-58.392592
2,Obelisco,Av. Corrientes 1066,CABA,None,-34.603824,-58.381759
3,Catedral de Córdoba,Independencia 30,Córdoba,Córdoba,-31.416382,-64.184089
4,Monumento Bandera,Av. Belgrano 1700,Santa Fe,Rosario,-32.960911,-60.621177
5,Catedral de La Plata,Calle 14 esq. 51,Buenos Aires,La Plata,-34.797681,-58.177080
6,Cerro de la Gloria,Av. del Libertador,Mendoza,Mendoza,-32.883735,-68.888632


## 4. Plot sobre Argenmap

Combinamos lo que devolvió Georef con el basemap argentino oficial.

In [5]:
m = folium.Map(location=[-34.6, -60], zoom_start=5, tiles=None)
basemaps.add_argenmap(m)

for _, row in df.dropna(subset=['lat', 'lon']).iterrows():
    popup = (
        f"<b>{row['nombre']}</b><br>"
        f"{row['direccion']}<br>"
        f"{row['localidad'] or ''} ({row['provincia']})<br>"
        f"<code>{row['lat']:.5f}, {row['lon']:.5f}</code>"
    )
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup, max_width=280),
        tooltip=row['nombre'],
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(m)

basemaps.add_creditos_argentina(m)
basemaps.add_layer_control(m)
m

## 5. Importancia del filtro por provincia/localidad

Muchas calles se repiten en distintas localidades. Si no filtrás, Georef te devuelve la primera que encuentre — puede no ser la que querías.

In [6]:
# Sin filtro
sin_filtro = direcciones.georreferenciar(direccion="Av. San Martín 100")
print("Sin filtro:")
print(" ", sin_filtro['nomenclatura'] if sin_filtro else 'sin resultado')
print()

for prov in ['CABA', 'Córdoba', 'Mendoza', 'Salta', 'Tucumán']:
    r = direcciones.georreferenciar(direccion="Av. San Martín 100", provincia=prov)
    print(f"Con provincia={prov}:")
    print(" ", r['nomenclatura'] if r else 'sin resultado')

Sin filtro:
  AV SAN MARTIN 100, Las Colonias, Santa Fe

Con provincia=CABA:
  sin resultado
Con provincia=Córdoba:
  AV LDOR GRL SAN MARTIN 100, General San Martín, Córdoba
Con provincia=Mendoza:
  AV GRL J DE SAN MARTIN 100, Luján de Cuyo, Mendoza
Con provincia=Salta:
  AV GRL SAN MARTIN 100, General José de San Martín, Salta
Con provincia=Tucumán:
  sin resultado


## 6. Dirección inexistente

Si Georef no encuentra coincidencias, las funciones devuelven `None` (sin romper).

In [7]:
r = direcciones.georreferenciar(direccion="calle inexistente 99999", provincia="CABA")
coords = direcciones.coordenadas(direccion="calle inexistente 99999", provincia="CABA")
print('georreferenciar →', r)
print('coordenadas    →', coords)

georreferenciar → None
coordenadas    → None


## Notas

- `argentina.direcciones` (sin `geo`) es parser local — limpia y separa calle/altura sin red.
- `argentina.geo.direcciones` consulta Georef y devuelve coordenadas reales + provincia/depto/localidad normalizados.
- Pasar `provincia=` y `localidad=` reduce drásticamente los falsos positivos en calles comunes.
- Para visualizar lo geocodeado, combinar con `argentina.geo.basemaps.add_argenmap(m)` mantiene la toponimia argentina (Islas Malvinas, sector antártico, etc.).